In [67]:
import pandas as pd
import sys
sys.path.append("../src/process_results")

from tqdm.auto import tqdm
tqdm.pandas()

from source_identification import classify_link_peer_review, extract_labels_per_row, label_distribution
from utils import fix_exists_by_status

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv('../data/results_gpt_4omini.csv')
df.head()

,prompt,result,references,tokens
0,"I want to write an article about: ""Common fair...",Several studies demonstrate the mathematical i...,['https://jmlr.org/beta/papers/v24/22-1511.htm...,NaN
1,"I want to write an article about: ""Machine Lea...",Machine learning models can exhibit bias even ...,['https://www.forbes.com/sites/aparnadhinakara...,NaN
2,"I want to write an article about: ""Evaluation ...",Evaluating and mitigating fairness solely thro...,['https://arxiv.org/abs/2006.09663?utm_source=...,NaN
3,"I want to write an article about: ""Benchmark c...",Several studies highlight how US-centric bench...,['https://davidnowak.me/why-your-ai-benchmarks...,NaN
4,"I want to write an article about: ""Word embedd...",Several studies have demonstrated that word em...,['https://arxiv.org/abs/1903.03862?utm_source=...,NaN


In [ ]:
df['result'].iloc[56]

In [4]:
import re
from urllib.parse import urlsplit, urlunsplit
import pandas as pd

# Regex general para URLs (http/https), evita capturar paréntesis/llaves/corchetes al final
_URL_RE = re.compile(r'https?://[^\s<>"\']+')

def _clean_url(u: str) -> str:
    """
    Limpia basura común al final: ), ], }, ., ,, ;, :
    y normaliza levemente.
    """
    u = u.strip()

    # Recorta cierres típicos que se pegan en Markdown o puntuación final
    while u and u[-1] in ')]}.,;:':
        u = u[:-1]

    # Opcional: normaliza (sin tocar query/utm)
    # (esto evita cosas raras como espacios u otros, pero es suave)
    parts = urlsplit(u)
    return urlunsplit(parts)

def extract_urls_from_text(text) -> list[str]:
    """
    Devuelve una lista (orden de aparición) con todas las URLs encontradas en el texto.
    Maneja NaN/None.
    """
    if text is None:
        return []
    # Pandas puede traer NaN (float)
    if isinstance(text, float) and pd.isna(text):
        return []

    s = str(text)

    urls = []
    for m in _URL_RE.finditer(s):
        urls.append(_clean_url(m.group(0)))

    # De-duplicar manteniendo orden
    seen = set()
    out = []
    for u in urls:
        if u and u not in seen:
            seen.add(u)
            out.append(u)
    return out

# --- Ejemplo de uso con DataFrame ---
# df["urls"] tendrá una lista por fila con todas las URLs encontradas en df["texto"]
# df["urls"] = df["texto"].apply(extract_urls_from_text)

# Si quieres una sola lista con todas las URLs del dataframe (aplanada):
# all_urls = [u for lst in df["urls"] for u in lst]

In [5]:
df["urls_clean"] = df["result"].apply(extract_urls_from_text)

In [6]:
df['urls_clean'].iloc[56]

['https://www.axios.com/2025/02/11/ai-security-revamp-def-con?utm_source=openai',
 'https://www.axios.com/2025/06/26/accenture-executives-cybersecurity-ai-plans?utm_source=openai',
 'https://www.crowdstrike.com/en-us/press-releases/crowdstrike-launches-ai-red-team-services-secure-ai-systems/?utm_source=openai',
 'https://learn.microsoft.com/en-us/security/ai-red-team/training?utm_source=openai',
 'https://arxiv.org/abs/2510.02677?utm_source=openai',
 'https://www.itpro.com/technology/artificial-intelligence/openai-turns-to-red-teamers-to-prevent-malicious-chatgpt-use-as-company-warns-future-models-could-pose-high-security-risk?utm_source=openai']

In [ ]:
df["url_labels"] = df["urls_clean"].progress_apply(lambda lst: [classify_link_peer_review(u) for u in lst])

In [ ]:
df.url_labels.iloc[56]

In [ ]:
df["labels"] = df["url_labels"].progress_apply(extract_labels_per_row)

In [ ]:
dist_urls = label_distribution(df, "url_labels", by="url")

In [ ]:
dist_urls

In [7]:
df['urls_clean'].iloc[2132]

['https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6131311/',
 'https://www.frontiersin.org/articles/10.3389/fpsyt.2020.00701/full']

In [ ]:
# TODO: Revisar las etiquetas unknown y ver cómo depuramos que sea peer reviewd o no.
# MISTRAL : Mirar referencias falsas por medio de google scholar (podemos comparar con algún cálculo de diferencia
# la cita en APA con la referencia que arroja el modelo - esto para los que aparecen en scholar pero puede pasar que
# haya referencias que ni el título exista)

In [ ]:
from urllib.parse import urlsplit

def base_url(url: str) -> str:
    """scheme + netloc. Ej: https://www.economist.com"""
    if not url:
        return ""
    p = urlsplit(str(url))
    if not p.scheme or not p.netloc:
        return ""
    return f"{p.scheme}://{p.netloc}"

def extract_baseurl_label_pairs_from_lists(
    df: pd.DataFrame,
    urls_col: str = "urls_clean",
    labels_col: str = "labels",
) -> pd.DataFrame:
    """
    df[urls_col]: lista de urls por fila
    df[labels_col]: lista de labels por fila (misma longitud idealmente)
    Devuelve DF largo con columnas: base_url, label
    """
    rows = []

    for urls, labels in zip(df[urls_col], df[labels_col]):
        if not isinstance(urls, list) or not isinstance(labels, list):
            continue

        # zip tolerante: corta al mínimo si vienen desalineadas
        for u, lab in zip(urls, labels):
            b = base_url(u)
            if b:
                rows.append({"base_url": b, "label": lab if lab is not None else "unknown"})

    return pd.DataFrame(rows, columns=["base_url", "label"])

def unique_labels_by_baseurl(pairs_df: pd.DataFrame) -> pd.DataFrame:
    """
    Agrupa por base_url y devuelve labels únicos por base_url.
    """
    if pairs_df.empty:
        return pd.DataFrame(columns=["base_url", "unique_labels", "n_unique_labels"])

    out = (
        pairs_df.groupby("base_url")["label"]
        .agg(lambda s: sorted(set(s)))
        .reset_index(name="unique_labels")
    )
    out["n_unique_labels"] = out["unique_labels"].apply(len)
    return out



In [ ]:
# --- Uso ---
pairs = extract_baseurl_label_pairs_from_lists(df, urls_col="urls_clean", labels_col="labels")
unique_map = unique_labels_by_baseurl(pairs)

inconsistent = unique_map[unique_map["n_unique_labels"] > 1].sort_values("n_unique_labels", ascending=False)

# pairs -> todas las (base_url, label)
# unique_map -> base_url con labels únicos

In [ ]:
inconsistent

In [ ]:
unique_map

In [ ]:
unique_map[unique_map.unique_labels.apply(lambda x: x[0]=='unknown')]

In [14]:
import requests
import pandas as pd

def url_exists(url, timeout: int = 15, allow_redirects: bool = True,
               accept_3xx: bool = False, max_bytes: int = 2048) -> dict:
    """
    Valida acceso real a la URL completa (GET, no solo HEAD).
    """
    # Normaliza / filtra NaN y valores raros
    if url is None:
        return {"url": url, "exists": False, "status_code": None, "final_url": None, "error": "empty"}
    if isinstance(url, float) and pd.isna(url):
        return {"url": None, "exists": False, "status_code": None, "final_url": None, "error": "empty"}
    if not isinstance(url, str):
        url = str(url)

    url = url.strip()
    if url == "" or url.lower() == "nan":
        return {"url": url, "exists": False, "status_code": None, "final_url": None, "error": "empty"}

    try:
        with requests.get(
            url,
            timeout=timeout,
            allow_redirects=allow_redirects,
            headers={"User-Agent": "url-validator/1.0"},
            stream=True,
        ) as r:
            status = r.status_code

            # Lee un poquito para confirmar acceso sin bajar todo
            try:
                for chunk in r.iter_content(chunk_size=max_bytes):
                    if chunk:
                        break
            except Exception:
                pass

            if 200 <= status < 300:
                exists = True
            elif accept_3xx and 300 <= status < 400:
                exists = True
            else:
                exists = False

            return {
                "url": url,
                "exists": exists,
                "status_code": status,
                "final_url": str(r.url),
                "error": None,
            }

    except requests.exceptions.RequestException as e:
        return {
            "url": url,
            "exists": False,
            "status_code": None,
            "final_url": None,
            "error": f"{type(e).__name__}: {e}",
        }

def url_exists_list(urls, **kwargs) -> list[dict]:
    """
    Aplica url_exists a una lista, tolerante a NaN y strings serializados.
    """
    urls = to_list_safe(urls)  # <- usa la función de arriba
    return [url_exists(u, **kwargs) for u in urls if u not in (None, "")]

In [14]:
ress = df["urls_clean"][:5].progress_apply(lambda lst: url_exists_list(lst, timeout=15))

100%|██████████| 5/5 [00:07<00:00,  1.51s/it]


In [18]:
ress[4]

[{'url': 'https://arxiv.org/abs/1903.03862?utm_source=openai',
  'exists': True,
  'status_code': 200,
  'final_url': 'https://arxiv.org/abs/1903.03862?utm_source=openai',
  'error': None},
 {'url': 'https://arxiv.org/abs/2005.00965?utm_source=openai',
  'exists': True,
  'status_code': 200,
  'final_url': 'https://arxiv.org/abs/2005.00965?utm_source=openai',
  'error': None},
 {'url': 'https://arxiv.org/abs/2203.13369?utm_source=openai',
  'exists': True,
  'status_code': 200,
  'final_url': 'https://arxiv.org/abs/2203.13369?utm_source=openai',
  'error': None},
 {'url': 'https://arxiv.org/abs/2112.05194?utm_source=openai',
  'exists': True,
  'status_code': 200,
  'final_url': 'https://arxiv.org/abs/2112.05194?utm_source=openai',
  'error': None},
 {'url': 'https://pubmed.ncbi.nlm.nih.gov/33400629/?utm_source=openai',
  'exists': True,
  'status_code': 200,
  'final_url': 'https://pubmed.ncbi.nlm.nih.gov/33400629/?utm_source=openai',
  'error': None},
 {'url': 'https://pubmed.ncbi.nl

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2135 entries, 0 to 2134
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   prompt      2135 non-null   object 
 1   result      2135 non-null   object 
 2   references  2135 non-null   object 
 3   tokens      0 non-null      float64
 4   urls_clean  2135 non-null   object 
dtypes: float64(1), object(4)
memory usage: 83.5+ KB


In [27]:
empty_res = df[df.references == '[]']#.to_csv('../data/empty_references_gpt.csv', index=False)
empty_res.drop(columns = ['result', 'references', 'tokens', 'urls_clean'], inplace=True)
empty_res.to_csv('../data/empty_references_gpt.csv', index=False)

/var/folders/t9/n67k3lts6wl76ntfbg_ntw3w0000gn/T/ipykernel_891/2982917392.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  empty_res.drop(columns = ['result', 'references', 'tokens', 'urls_clean'], inplace=True)


In [28]:
df["url_checks"] = df["urls_clean"].progress_apply(lambda lst: url_exists_list(lst, timeout=15))

  0%|          | 0/2135 [00:00<?, ?it/s]

100%|██████████| 2135/2135 [1:34:40<00:00,  2.66s/it]  


In [32]:
df.to_csv('../data/processed/gpt_url_checks.csv', index=False)

In [3]:
df_wb = pd.read_csv('../data/results_gpt_web_search.csv')
df_wb.head()

,prompt,result,references,tokens
0,"Explain the following statement: ""Common fairn...","The statement ""Common fairness definitions are...",['https://algorithmaudit.eu/knowledge-platform...,NaN
1,"Explain the following statement: ""Machine Lear...",Machine learning models can exhibit bias even ...,['https://www.forbes.com/sites/aparnadhinakara...,NaN
2,"Explain the following statement: ""Evaluation a...",Evaluating and mitigating fairness solely thro...,['https://arxiv.org/abs/2006.09663?utm_source=...,NaN
3,"Explain the following statement: ""Benchmark ch...",Benchmark choices for Generative AI (GenAI) mo...,['https://www.tomsguide.com/ai/ai-safety-tests...,NaN
4,"Explain the following statement: ""Word embeddi...","Word embeddings, which are vector representati...",['https://pubmed.ncbi.nlm.nih.gov/33400629/?ut...,NaN


In [4]:
df_wb[df_wb.references == '[]']

,prompt,result,references,tokens
717,Evaluate the strength of evidence across the f...,Data centers utilize water both directly for l...,[],NaN


In [ ]:
# TODO: para los que no están en peer reviewd, revisar si existe su versión publicada
df.head()

In [11]:
import ast
import pandas as pd

def to_list_safe(x):
    """
    Convierte:
    - NaN/None -> []
    - lista -> lista
    - string tipo "['a','b']" -> lista
    - string 'nan' / '' -> []
    - cualquier otro -> []
    """
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        s = x.strip()
        if s == "" or s.lower() == "nan":
            return []
        # si parece lista serializada
        if s.startswith("[") and s.endswith("]"):
            try:
                return ast.literal_eval(s)
            except (ValueError, SyntaxError):
                return []
        return []
    return []

# Ejemplo:
# df["urls_clean"] = df["urls_clean"].apply(to_list_safe)

In [13]:
df_wb["urls_clean"] = df_wb["references"].apply(to_list_safe)

In [16]:
df_wb["url_checks"] = df_wb["urls_clean"].progress_apply(lambda lst: url_exists_list(lst, timeout=10))

100%|██████████| 1042/1042 [4:40:21<00:00, 16.14s/it] 


In [17]:
df_wb.to_csv('../data/processed/gpt_web_search_url_checks.csv', index=False)

In [ ]:
df_wb.references.iloc[0]

"['https://algorithmaudit.eu/knowledge-platform/knowledge-base/measure_mismeasure_fairness/?utm_source=openai', 'https://jmlr.org/beta/papers/v24/22-1511.html?utm_source=openai', 'https://ieeexplore.ieee.org/document/9622861?utm_source=openai', 'https://www.hks.harvard.edu/publications/measure-and-mismeasure-fairness?utm_source=openai', 'https://developers.google.com/machine-learning/glossary/fairness?utm_source=openai', 'https://www.emergentmind.com/papers/1609.07236?utm_source=openai', 'https://simons.berkeley.edu/talks/measure-mismeasure-fairness?utm_source=openai', 'https://link.springer.com/article/10.1007/s13347-024-00814-z?utm_source=openai', 'https://innovation.world/invention/fairness-impossibility-theorem-machine-learning/?utm_source=openai', 'https://mlhp.stanford.edu/src/chap6.html?utm_source=openai', 'https://proceedings.mlr.press/v162/nilforoshan22a.html?utm_source=openai', 'https://fairlearn.org/v0.7.0/user_guide/fairness_in_machine_learning.html?utm_source=openai']"

In [38]:
type(df["urls_clean"].iloc[4])

list

In [1]:
import pandas as pd

In [2]:
df1 = pd.read_csv('../data/processed/gpt_web_search_url_checks.csv')
df1.head()

,prompt,result,references,tokens,urls_clean,url_checks
0,"Explain the following statement: ""Common fairn...","The statement ""Common fairness definitions are...",['https://algorithmaudit.eu/knowledge-platform...,NaN,['https://algorithmaudit.eu/knowledge-platform...,[{'url': 'https://algorithmaudit.eu/knowledge-...
1,"Explain the following statement: ""Machine Lear...",Machine learning models can exhibit bias even ...,['https://www.forbes.com/sites/aparnadhinakara...,NaN,['https://www.forbes.com/sites/aparnadhinakara...,[{'url': 'https://www.forbes.com/sites/aparnad...
2,"Explain the following statement: ""Evaluation a...",Evaluating and mitigating fairness solely thro...,['https://arxiv.org/abs/2006.09663?utm_source=...,NaN,['https://arxiv.org/abs/2006.09663?utm_source=...,[{'url': 'https://arxiv.org/abs/2006.09663?utm...
3,"Explain the following statement: ""Benchmark ch...",Benchmark choices for Generative AI (GenAI) mo...,['https://www.tomsguide.com/ai/ai-safety-tests...,NaN,['https://www.tomsguide.com/ai/ai-safety-tests...,[{'url': 'https://www.tomsguide.com/ai/ai-safe...
4,"Explain the following statement: ""Word embeddi...","Word embeddings, which are vector representati...",['https://pubmed.ncbi.nlm.nih.gov/33400629/?ut...,NaN,['https://pubmed.ncbi.nlm.nih.gov/33400629/?ut...,[{'url': 'https://pubmed.ncbi.nlm.nih.gov/3340...


In [27]:
df1.shape

(1042, 9)

In [3]:
df2 = pd.read_csv('../data/processed/gpt_url_checks.csv')
df2.head()

,prompt,result,references,tokens,urls_clean,url_checks
0,"I want to write an article about: ""Common fair...",Several studies demonstrate the mathematical i...,['https://jmlr.org/beta/papers/v24/22-1511.htm...,NaN,['https://jmlr.org/beta/papers/v24/22-1511.htm...,[{'url': 'https://jmlr.org/beta/papers/v24/22-...
1,"I want to write an article about: ""Machine Lea...",Machine learning models can exhibit bias even ...,['https://www.forbes.com/sites/aparnadhinakara...,NaN,['https://www.forbes.com/sites/aparnadhinakara...,[{'url': 'https://www.forbes.com/sites/aparnad...
2,"I want to write an article about: ""Evaluation ...",Evaluating and mitigating fairness solely thro...,['https://arxiv.org/abs/2006.09663?utm_source=...,NaN,['https://arxiv.org/abs/2006.09663?utm_source=...,[{'url': 'https://arxiv.org/abs/2006.09663?utm...
3,"I want to write an article about: ""Benchmark c...",Several studies highlight how US-centric bench...,['https://davidnowak.me/why-your-ai-benchmarks...,NaN,['https://davidnowak.me/why-your-ai-benchmarks...,[{'url': 'https://davidnowak.me/why-your-ai-be...
4,"I want to write an article about: ""Word embedd...",Several studies have demonstrated that word em...,['https://arxiv.org/abs/1903.03862?utm_source=...,NaN,['https://arxiv.org/abs/1903.03862?utm_source=...,[{'url': 'https://arxiv.org/abs/1903.03862?utm...


In [4]:
df1.url_checks.iloc[0]

"[{'url': 'https://algorithmaudit.eu/knowledge-platform/knowledge-base/measure_mismeasure_fairness/?utm_source=openai', 'exists': False, 'status_code': 404, 'final_url': 'https://algorithmaudit.eu/knowledge-platform/knowledge-base/measure_mismeasure_fairness/?utm_source=openai', 'error': None}, {'url': 'https://jmlr.org/beta/papers/v24/22-1511.html?utm_source=openai', 'exists': True, 'status_code': 200, 'final_url': 'https://jmlr.org/beta/papers/v24/22-1511.html?utm_source=openai', 'error': None}, {'url': 'https://ieeexplore.ieee.org/document/9622861?utm_source=openai', 'exists': True, 'status_code': 200, 'final_url': 'https://ieeexplore.ieee.org/document/9622861?utm_source=openai', 'error': None}, {'url': 'https://www.hks.harvard.edu/publications/measure-and-mismeasure-fairness?utm_source=openai', 'exists': False, 'status_code': 403, 'final_url': 'https://www.hks.harvard.edu/publications/measure-and-mismeasure-fairness?utm_source=openai', 'error': None}, {'url': 'https://developers.go

In [7]:
import ast
def extract_status_codes(url_checks):
    """
    Dado url_checks (lista de dicts), devuelve lista de status_code.
    Maneja NaN/None.
    """
    if url_checks is None:
        return []
    if isinstance(url_checks, float) and pd.isna(url_checks):
        return []
    if isinstance(url_checks, str):
        try:
            url_checks = ast.literal_eval(url_checks)
        except (ValueError, SyntaxError):
            return []

    if not isinstance(url_checks, list):
        return []

    status_codes = []
    for check in url_checks:
        if isinstance(check, dict) and "status_code" in check:
            status_codes.append(check["exists"])
    return status_codes

In [73]:
df1["url_checks_clean"] = df1["url_checks"].apply(fix_exists_by_status)
df2["url_checks_clean"] = df2["url_checks"].apply(fix_exists_by_status)

In [78]:
df1['status_codes'] = df1['url_checks_clean'].apply(extract_status_codes)

In [79]:
df2['status_codes'] = df2['url_checks_clean'].apply(extract_status_codes)

In [14]:
def get_total(lista):
    return lista.count(False), lista.count(True)

In [80]:
df1['Total_False'], df1['Total_True'] = zip(*df1['status_codes'].apply(get_total))

In [81]:
df2['Total_False'], df2['Total_True'] = zip(*df2['status_codes'].apply(get_total))

In [82]:
df1.Total_False.sum(), df1.Total_True.sum(),  df1.Total_False.sum() + df1.Total_True.sum(), df1.shape

(np.int64(728), np.int64(14140), np.int64(14868), (1042, 10))

In [83]:
df2.Total_False.sum(), df2.Total_True.sum(),  df2.Total_False.sum() + df2.Total_True.sum(), df2.shape

(np.int64(1749), np.int64(5528), np.int64(7277), (2135, 10))

In [24]:
df2.shape, df1.shape

((2135, 9), (1042, 9))

In [66]:
for i in codes:
    code = i
    print(code, true_urls_df1[status_codes_df1.index(code)], existe_no[status_codes_df1.index(code)])

200 https://jmlr.org/beta/papers/v24/22-1511.html?utm_source=openai True
521 https://www.shaunstoltz.com/2025/03/06/managing-risks-in-internal-audit-outsourcing-a-comprehensive-guide/?utm_source=openai False
202 https://forums.autodesk.com/t5/-/-/m-p/13490316?utm_source=openai True
429 https://www.bincial.com/news/tzArtificialIntelligence/138534?utm_source=openai False
302 https://www.peeref.com/works/85257961?utm_source=openai False
525 https://logiccheck.ai/logical-fallacy/appeal-to-simplicity-unraveling-the-bias-in-oversimplified-thinking/?utm_source=openai False
401 https://springerlink.fh-diploma.de/doi/10.1007/s00146-020-00960-w?utm_source=openai False
402 https://www.thoughtco.com/principle-of-least-effort-zipfs-law-1691104?utm_source=openai False
403 https://www.hks.harvard.edu/publications/measure-and-mismeasure-fairness?utm_source=openai False
404 https://algorithmaudit.eu/knowledge-platform/knowledge-base/measure_mismeasure_fairness/?utm_source=openai False
500 https://gener

In [ ]:
import ast

def extract_existing_urls(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """
    Extrae todas las URLs con exists=True de una columna con listas de dicts,
    y retorna un DataFrame donde cada URL es una fila.
    """
    rows = []

    for idx, cell_value in df[col].items():
        try:
            records = ast.literal_eval(cell_value)
        except (ValueError, SyntaxError):
            continue

        for record in records:
            if record.get("exists") is True:
                rows.append({
                    "original_row": idx,        # índice original del df, útil para trazabilidad
                    "url": record.get("url"),
                    "status_code": record.get("status_code"),
                    "final_url": record.get("final_url"),
                    "error": record.get("error"),
                })

    return pd.DataFrame(rows)


In [84]:
# Usar
urls_df = extract_existing_urls(df1, "url_checks_clean")
urls_df2 = extract_existing_urls(df2, "url_checks_clean")

In [85]:
urls_df

,original_row,url,status_code,final_url,error
0,0,https://jmlr.org/beta/papers/v24/22-1511.html?...,200,https://jmlr.org/beta/papers/v24/22-1511.html?...,None
1,0,https://ieeexplore.ieee.org/document/9622861?u...,200,https://ieeexplore.ieee.org/document/9622861?u...,None
2,0,https://www.hks.harvard.edu/publications/measu...,403,https://www.hks.harvard.edu/publications/measu...,None
3,0,https://developers.google.com/machine-learning...,200,https://developers.google.com/machine-learning...,None
4,0,https://www.emergentmind.com/papers/1609.07236...,200,https://www.emergentmind.com/papers/1609.07236...,None
...,...,...,...,...,...
14135,1041,https://quickonomics.com/terms/rebound-effect/...,200,https://quickonomics.com/terms/rebound-effect/...,None
14136,1041,https://www.collinsdictionary.com/dictionary/e...,403,https://www.collinsdictionary.com/dictionary/e...,None
14137,1041,https://www.sciencedirect.com/topics/immunolog...,403,https://www.sciencedirect.com/topics/immunolog...,None
14138,1041,https://sustainability.bonares.de/sustainabili...,503,https://sustainability.bonares.de/sustainabili...,None


In [86]:
urls_df2

,original_row,url,status_code,final_url,error
0,0,https://jmlr.org/beta/papers/v24/22-1511.html?...,200,https://jmlr.org/beta/papers/v24/22-1511.html?...,None
1,0,https://arxiv.org/abs/1902.04783?utm_source=op...,200,https://arxiv.org/abs/1902.04783?utm_source=op...,None
2,0,https://arxiv.org/abs/1711.05144?utm_source=op...,200,https://arxiv.org/abs/1711.05144?utm_source=op...,None
3,0,https://arxiv.org/abs/2107.04642?utm_source=op...,200,https://arxiv.org/abs/2107.04642?utm_source=op...,None
4,0,https://www.emergentmind.com/papers/1609.07236...,200,https://www.emergentmind.com/papers/1609.07236...,None
...,...,...,...,...,...
5523,2133,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5...,200,https://pmc.ncbi.nlm.nih.gov/articles/PMC5571865/,None
5524,2133,https://www.ncbi.nlm.nih.gov/books/NBK499877/,200,https://www.ncbi.nlm.nih.gov/books/NBK499877/,None
5525,2134,https://www.economist.com/schools-brief/2019/0...,403,https://www.economist.com/schools-brief/2019/0...,None
5526,2134,https://www.sciencedirect.com/science/article/...,403,https://www.sciencedirect.com/science/article/...,None


In [87]:
urls_df["url"].to_csv("urls1.txt", index=False, header=False)
urls_df2["url"].to_csv("urls2.txt", index=False, header=False)

In [88]:
from classify_urls import process_urls

urls_list = urls_df["url"].tolist()
results = process_urls(urls_list, workers=8)

classified_df = pd.DataFrame(results)

[██████████████████████████████] 14140/14140  


In [89]:
urls_list2 = urls_df2["url"].tolist()
results2 = process_urls(urls_list2, workers=8)

classified_df2 = pd.DataFrame(results2)

[██████████████████████████████] 5528/5528  


In [92]:
classified_df.peer_reviewed.value_counts(dropna=False)

peer_reviewed
no    9273
sí    4867
Name: count, dtype: int64

In [93]:
classified_df2.peer_reviewed.value_counts(dropna=False)

peer_reviewed
no    3770
sí    1758
Name: count, dtype: int64

In [ ]:
g